In [2]:
import pandas as pd
import glob

folder = './barcode22/filtered/combined/'
report_ext = '*.report'
krakenout_ext = '*.kraken'

# search folder for files with the specified extensions
report_files = glob.glob(folder + report_ext)
krakenout_files = glob.glob(folder + krakenout_ext) 

for file in report_files:
    print('reports: ',file)

for file in krakenout_files:
    print('Kraken outputs: ',file)

reports:  ./barcode22/filtered/combined/combined_0_unique.fasta.report
reports:  ./barcode22/filtered/combined/combined_1_unique.fasta.report
reports:  ./barcode22/filtered/combined/combined_2_unique.fasta.report
reports:  ./barcode22/filtered/combined/combined_3_unique.fasta.report
reports:  ./barcode22/filtered/combined/combined_5_unique.fasta.report
reports:  ./barcode22/filtered/combined/combined_6_unique.fasta.report
reports:  ./barcode22/filtered/combined/combined_consensussequences.fasta.report
reports:  ./barcode22/filtered/combined/combined_nogroup_unique.fasta.report
Kraken outputs:  ./barcode22/filtered/combined/combined_0_unique.fasta.kraken
Kraken outputs:  ./barcode22/filtered/combined/combined_1_unique.fasta.kraken
Kraken outputs:  ./barcode22/filtered/combined/combined_2_unique.fasta.kraken
Kraken outputs:  ./barcode22/filtered/combined/combined_3_unique.fasta.kraken
Kraken outputs:  ./barcode22/filtered/combined/combined_5_unique.fasta.kraken
Kraken outputs:  ./barcode

In [ ]:
columns = ['status','readID','taxonID','length','LCA_list']
krakenout_frames = []
for file in krakenout_files:
    temp = pd.read_csv(file, sep = '\t', header = None, names = columns)
    if not temp.empty:
        krakenout_frames.append(temp)
krakenout_concat = pd.concat(krakenout_frames, ignore_index = True) if krakenout_frames else pd.DataFrame(columns = columns)

# percent in taxId is the percentage of reads covered by the clade rooted in the taxon (cummulative percentage of reads in the taxon and all its children)
# number of reads covered by the clade rooted in the taxon including child taxa (cummulative number of reads in the taxon and all its children)
# number of fragments in the taxon esclusively and not including child taxa
# A rank code:
# U = Unclassified, R = Root, D = Domain, P = Phylum, C = Class, O = Order, F = Family, G = Genus, S = Species
# NCBI taxID
# originally indented taxonomic name
columns = ['percent_in_taxID','reads_in_hierarchy','num_in_taxID','rank','taxID','tax_name']
report_frames = []
for file in report_files:
    temp = pd.read_csv(file, sep = '\t', header = None, names = columns)
    # remove multiple spaces after \t but keep single spaces in species names
    temp['tax_name'] = temp['tax_name'].str.strip()
    if not temp.empty:
        report_frames.append(temp)

reports_concat = pd.concat(report_frames, ignore_index=True) if report_frames else pd.DataFrame(columns=columns)

reports_concat


,percent_in_taxID,reads_in_hierarchy,num_in_taxID,rank,taxID,tax_name
0,100.0,2,2,U,0,unclassified
1,100.0,10,10,U,0,unclassified
2,100.0,2,2,U,0,unclassified
3,100.0,105,0,R,1,root
4,100.0,105,0,R1,131567,cellular organisms
...,...,...,...,...,...,...
8732,0.0,2,2,S,340016,uncultured virus
8733,0.0,2,0,R1,2787854,other entries
8734,0.0,2,0,R2,28384,other sequences
8735,0.0,2,0,R3,81077,artificial sequences


In [9]:
# dictionary linking taxonomic name and taxonomic level
taxID_to_rank = dict(zip(reports_concat['taxID'], reports_concat['rank']))
taxID_to_rank

taxID_to_species = dict(zip(reports_concat['taxID'], reports_concat['tax_name']))
taxID_to_species   


{0: 'unclassified',
 1: 'root',
 131567: 'cellular organisms',
 2759: 'Eukaryota',
 33154: 'Opisthokonta',
 33208: 'Metazoa',
 6072: 'Eumetazoa',
 33213: 'Bilateria',
 33317: 'Protostomia',
 1206794: 'Ecdysozoa',
 88770: 'Panarthropoda',
 6656: 'Arthropoda',
 197563: 'Mandibulata',
 197562: 'Pancrustacea',
 6960: 'Hexapoda',
 50557: 'Insecta',
 85512: 'Dicondylia',
 7496: 'Pterygota',
 33340: 'Neoptera',
 33392: 'Endopterygota',
 7147: 'Diptera',
 7148: 'Nematocera',
 43784: 'Bibionomorpha',
 41830: 'Sciaroidea',
 7184: 'Sciaridae',
 1260859: 'Scatopsciara',
 2890712: 'Scatopsciara brevicornis',
 1817658: 'Claustropyga',
 2881732: 'Claustropyga brevichaeta',
 265459: 'unclassified Sciaridae',
 2909299: 'Sciaridae sp.',
 170626: 'Lycoriella',
 2621320: 'unclassified Lycoriella',
 2405327: 'Lycoriella sp. BIOUG23129-C06',
 29035: 'Mycetophilidae',
 43800: 'Sciophilinae',
 1888027: 'Acnemia',
 1888028: 'Acnemia amoena',
 397112: 'Leiinae',
 397136: 'Docosia',
 2763083: 'Docosia rameli',
 

In [13]:
krakenout_concat['taxonName'] = krakenout_concat['taxonID'].map(taxID_to_species)
krakenout_concat['taxonRank'] = krakenout_concat['taxonID'].map(taxID_to_rank)
krakenout_concat.sort_values(by = 'taxonName', inplace = True)
krakenout_concat.to_csv('./barcode22/filtered/combined/krakenout_concat_b22_cleanfirst.csv', index = False)

In [15]:
krakenout_concat_bytaxonName = krakenout_concat.groupby('taxonName').size().reset_index(name='counts')
krakenout_concat_bytaxonName.sort_values(by = 'counts', ascending = False, inplace = True)
krakenout_concat_bytaxonName.to_csv('./barcode22/filtered/combined/krakenout_concat_bytaxonName_b22_cleanfirst_bytaxonName.csv', index = False)